# 📐 Chapter 2：線性組合、張成的空間與基

> **3Blue1Brown — Essence of Linear Algebra**  
> Linear combinations, span, and basis

本筆記對應 3Blue1Brown《線性代數的本質》第 2 章。  
我們將探討：
1. **線性組合 (Linear Combination)** — 用純量縮放向量再相加
2. **張成的空間 (Span)** — 所有線性組合的集合
3. **基底 (Basis)** — 能張成整個空間的最小線性獨立集
4. **線性相依 vs 線性獨立** — 向量之間是否有「多餘」的關係

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Helper functions ──────────────────────────────────────────────

def setup_ax(ax, title="", xlim=(-6, 6), ylim=(-6, 6)):
    """Configure a clean 2D axis with grid, equal aspect, and origin axes."""
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.grid(True, linewidth=0.3, alpha=0.5)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    if title:
        ax.set_title(title, fontsize=13, fontweight="bold")


def draw_vec(ax, vec, origin=(0, 0), color="black", label=None, lw=2, alpha=1.0):
    """Draw an arrow from origin to origin+vec with an optional label."""
    ax.annotate(
        "",
        xy=(origin[0] + vec[0], origin[1] + vec[1]),
        xytext=origin,
        arrowprops=dict(arrowstyle="->,head_width=0.25,head_length=0.2",
                        color=color, lw=lw, alpha=alpha),
    )
    if label:
        mid_x = origin[0] + vec[0] / 2
        mid_y = origin[1] + vec[1] / 2
        ax.text(mid_x + 0.15, mid_y + 0.15, label,
                fontsize=10, color=color, fontweight="bold", alpha=alpha)

print("Helper functions ready ✓")

---
## Part 1：線性組合 (Linear Combination)

$$a\vec{v} + b\vec{w}$$

用兩個**純量** (scalar) $a$, $b$ 分別縮放兩個向量 $\vec{v}$, $\vec{w}$，再相加，得到的結果就是一個**線性組合**。

- 固定向量，改變純量 → 得到不同的線性組合
- 「線性」：每個向量只被乘以一個常數，沒有平方、開根號等非線性操作
- 當 $a + b = 1$ 時，結果落在 $\vec{v}$ 到 $\vec{w}$ 的連線上

In [ ]:
# ── Part 1: Visualize linear combination av + bw ─────────────────

v = np.array([1, 2])
w = np.array([3, 1])

# Try different (a, b) pairs
combos = [(1, 1), (2, -1), (0.5, 1.5), (-1, 2)]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, (a, b) in zip(axes, combos):
    setup_ax(ax, title=f"a={a}, b={b}", xlim=(-5, 8), ylim=(-4, 6))

    av = a * v
    bw = b * w
    result = av + bw

    # Draw original v and w (faded)
    draw_vec(ax, v, color="grey", label="v", lw=1.5, alpha=0.4)
    draw_vec(ax, w, color="grey", label="w", lw=1.5, alpha=0.4)

    # Draw av (from origin) then bw (from tip of av) — parallelogram rule
    draw_vec(ax, av, color="#2196F3", label=f"{a}v", lw=2)
    draw_vec(ax, bw, origin=tuple(av), color="#FF9800", label=f"{b}w", lw=2)

    # Draw result av + bw
    draw_vec(ax, result, color="#E91E63", label=f"av+bw", lw=2.5)

    # Mark the result point
    ax.plot(*result, "o", color="#E91E63", markersize=6)

fig.suptitle("線性組合 av + bw：固定 v=(1,2), w=(3,1)，改變 a, b",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Part 2：張成的空間 (Span)

所有 $a\vec{v} + b\vec{w}$（$a, b$ 取遍所有實數）的集合，就是 $\vec{v}$ 和 $\vec{w}$ 的**張成空間** (span)。

$$\text{span}(\vec{v}, \vec{w}) = \{a\vec{v} + b\vec{w} \mid a, b \in \mathbb{R}\}$$

三種情況：
| 情況 | span 的結果 |
|------|------------|
| $\vec{v}, \vec{w}$ **不平行**（且非零） | 整個 2D 平面 |
| $\vec{v}, \vec{w}$ **平行**（或其中一個為零） | 一條直線 |
| $\vec{v} = \vec{w} = \vec{0}$ | 只有原點 |

> 💡 **直覺**：兩個不平行的向量可以「拼出」平面上任何位置，就像兩個方向盤一樣。

In [ ]:
# ── Part 2: Span — non-parallel vs parallel vectors ──────────────

np.random.seed(42)
N = 2000  # number of random (a, b) samples
ab = np.random.uniform(-3, 3, size=(N, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# --- Case 1: Non-parallel vectors → span = full plane ---
v1 = np.array([1, 2])
w1 = np.array([3, 1])
points1 = ab[:, 0:1] * v1 + ab[:, 1:2] * w1  # (N, 2)

ax = axes[0]
setup_ax(ax, title="不平行：span = 整個平面", xlim=(-12, 12), ylim=(-10, 10))
ax.scatter(points1[:, 0], points1[:, 1], s=4, alpha=0.3, color="#2196F3")
draw_vec(ax, v1 * 2, color="#E91E63", label="v=(1,2)", lw=2.5)
draw_vec(ax, w1 * 2, color="#FF9800", label="w=(3,1)", lw=2.5)

# --- Case 2: Parallel vectors → span = a line ---
v2 = np.array([1, 2])
w2 = np.array([2, 4])  # w2 = 2 * v2, parallel!
points2 = ab[:, 0:1] * v2 + ab[:, 1:2] * w2

ax = axes[1]
setup_ax(ax, title="平行：span = 一條線", xlim=(-12, 12), ylim=(-10, 10))
ax.scatter(points2[:, 0], points2[:, 1], s=4, alpha=0.3, color="#FF9800")
draw_vec(ax, v2 * 2, color="#E91E63", label="v=(1,2)", lw=2.5)
draw_vec(ax, w2 * 1, color="#9C27B0", label="w=(2,4)", lw=2.5)

fig.suptitle("張成的空間 (Span)：用隨機 a,b 產生所有 av+bw",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3：基底 (Basis)

**基底** = 一組向量，滿足兩個條件：
1. **線性獨立** — 沒有多餘的向量
2. **張成整個空間** — 它們的 span 覆蓋整個空間

### 標準基底 (Standard Basis)

$$\hat{i} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}, \quad \hat{j} = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$$

任何向量都可以表示為基底的線性組合：

$$\begin{bmatrix} 3 \\ 2 \end{bmatrix} = 3\hat{i} + 2\hat{j}$$

> 💡 **座標的本質**：座標 $(3, 2)$ 其實就是在告訴你「用 3 倍的 $\hat{i}$ 加上 2 倍的 $\hat{j}$」。座標是相對於基底的線性組合係數！

### 非標準基底

如果換一組基底，同一個向量的「座標」也會改變。

In [ ]:
# ── Part 3: Express a vector using a non-standard basis ──────────

# New basis vectors
b1 = np.array([2, 1])
b2 = np.array([-1, 1])

# Target vector (in standard coordinates)
target = np.array([3, 2])

# Solve for coefficients: c1*b1 + c2*b2 = target
# This is  [b1 | b2] @ [c1, c2]^T = target
A = np.column_stack([b1, b2])
coeffs = np.linalg.solve(A, target)
c1, c2 = coeffs

print(f"基底 b1 = {b1}, b2 = {b2}")
print(f"目標向量 = {target}")
print(f"係數: c1 = {c1:.4f}, c2 = {c2:.4f}")
print(f"驗證: {c1:.4f}*b1 + {c2:.4f}*b2 = {c1*b1 + c2*b2}")

# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: standard basis ---
ax = axes[0]
setup_ax(ax, title="標準基底：3i + 2j", xlim=(-1, 5), ylim=(-1, 4))
draw_vec(ax, np.array([1, 0]) * 3, color="#2196F3", label="3i", lw=2)
draw_vec(ax, np.array([0, 1]) * 2, origin=(3, 0), color="#FF9800", label="2j", lw=2)
draw_vec(ax, target, color="#E91E63", label="(3,2)", lw=2.5)
ax.plot(*target, "o", color="#E91E63", markersize=8)

# --- Right: non-standard basis ---
ax = axes[1]
setup_ax(ax, title=f"新基底：{c1:.2f}·b1 + {c2:.2f}·b2", xlim=(-3, 6), ylim=(-1, 4))

# Draw scaled basis vectors with tip-to-tail
scaled_b1 = c1 * b1
scaled_b2 = c2 * b2
draw_vec(ax, scaled_b1, color="#2196F3", label=f"{c1:.2f}·b1", lw=2)
draw_vec(ax, scaled_b2, origin=tuple(scaled_b1), color="#FF9800", label=f"{c2:.2f}·b2", lw=2)
draw_vec(ax, target, color="#E91E63", label="target", lw=2.5)

# Draw basis directions (faded)
draw_vec(ax, b1, color="#2196F3", lw=1, alpha=0.3)
draw_vec(ax, b2, color="#FF9800", lw=1, alpha=0.3)
ax.plot(*target, "o", color="#E91E63", markersize=8)

fig.suptitle("基底 (Basis)：同一個向量，不同基底下的表示",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4：線性相依 vs 線性獨立

### 線性獨立 (Linearly Independent)
每個向量都**帶來新的方向**，移除任何一個都會讓 span 縮小。

$$a\vec{v} + b\vec{w} = \vec{0} \quad \Longrightarrow \quad a = b = 0$$

### 線性相依 (Linearly Dependent)
有一個向量是其他向量的**線性組合**（多餘的），移除它 span 不變。

$$\vec{w} = c \cdot \vec{v} \quad \text{（平行向量必線性相依）}$$

> 💡 **3Blue1Brown 的說法**：如果有一個向量可以被其他的向量「表達」，它就是多餘的——線性相依。

In [ ]:
# ── Part 4: Independent vs Dependent — side by side ──────────────

np.random.seed(7)
N = 1500
ab = np.random.uniform(-3, 3, size=(N, 2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: Linearly Independent ---
v_ind = np.array([1, 2])
w_ind = np.array([3, 1])
pts_ind = ab[:, 0:1] * v_ind + ab[:, 1:2] * w_ind

ax = axes[0]
setup_ax(ax, title="線性獨立 — span = 2D 平面", xlim=(-12, 12), ylim=(-10, 10))
ax.scatter(pts_ind[:, 0], pts_ind[:, 1], s=3, alpha=0.2, color="#2196F3")
draw_vec(ax, v_ind * 2, color="#E91E63", label="v", lw=2.5)
draw_vec(ax, w_ind * 2, color="#FF9800", label="w", lw=2.5)
ax.text(-11, 8, "dim(span) = 2", fontsize=12, fontweight="bold",
        color="#2196F3", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#2196F3"))

# --- Right: Linearly Dependent ---
v_dep = np.array([1, 2])
w_dep = np.array([-2, -4])  # w = -2v → dependent
pts_dep = ab[:, 0:1] * v_dep + ab[:, 1:2] * w_dep

ax = axes[1]
setup_ax(ax, title="線性相依 — span = 一條線", xlim=(-12, 12), ylim=(-10, 10))
ax.scatter(pts_dep[:, 0], pts_dep[:, 1], s=3, alpha=0.2, color="#FF9800")
draw_vec(ax, v_dep * 2, color="#E91E63", label="v", lw=2.5)
draw_vec(ax, w_dep * 1, color="#9C27B0", label="w=−2v", lw=2.5)
ax.text(-11, 8, "dim(span) = 1", fontsize=12, fontweight="bold",
        color="#FF9800", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#FF9800"))
ax.text(-11, 5.5, "w = −2v (多餘的!)", fontsize=10, color="#9C27B0")

fig.suptitle("線性獨立 vs 線性相依",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Part 5：練習區

### 練習 1：手動算線性組合
給定 $\vec{v} = (2, -1)$，$\vec{w} = (1, 3)$，計算 $3\vec{v} + (-2)\vec{w}$ = ?

### 練習 2：判斷線性相依
以下哪組向量是線性相依的？
- (A) $\vec{v} = (1, 3)$, $\vec{w} = (2, 6)$
- (B) $\vec{v} = (1, 0)$, $\vec{w} = (0, 1)$
- (C) $\vec{v} = (3, 1)$, $\vec{w} = (-1, 2)$

### 練習 3：用非標準基底表示向量
基底 $\vec{b_1} = (1, 1)$, $\vec{b_2} = (1, -1)$，求向量 $(5, 1)$ 在此基底下的座標。

In [ ]:
# ── Exercise 1: Manual linear combination ────────────────────────
v_ex = np.array([2, -1])
w_ex = np.array([1, 3])
result_ex1 = 3 * v_ex + (-2) * w_ex
print(f"練習 1: 3v + (-2)w = 3{list(v_ex)} + (-2){list(w_ex)} = {list(result_ex1)}")
# Answer: (4, -9)

# ── Exercise 2: Check linear dependence ──────────────────────────
pairs = [
    ("A", np.array([1, 3]), np.array([2, 6])),
    ("B", np.array([1, 0]), np.array([0, 1])),
    ("C", np.array([3, 1]), np.array([-1, 2])),
]
print("\n練習 2: 判斷線性相依")
for name, va, wa in pairs:
    # Check if determinant is zero → linearly dependent
    det = va[0] * wa[1] - va[1] * wa[0]
    status = "線性相依 ✗" if abs(det) < 1e-10 else "線性獨立 ✓"
    print(f"  ({name}) det = {det:>4d} → {status}")
# Answer: (A) is dependent — w = 2v

# ── Exercise 3: Change of basis ──────────────────────────────────
b1_ex = np.array([1, 1])
b2_ex = np.array([1, -1])
target_ex = np.array([5, 1])
A_ex = np.column_stack([b1_ex, b2_ex])
coeffs_ex = np.linalg.solve(A_ex, target_ex)
print(f"\n練習 3: 向量 {list(target_ex)} 在基底 b1={list(b1_ex)}, b2={list(b2_ex)} 下的座標")
print(f"  座標 = ({coeffs_ex[0]}, {coeffs_ex[1]})")
print(f"  驗證: {coeffs_ex[0]}*b1 + {coeffs_ex[1]}*b2 = {coeffs_ex[0]*b1_ex + coeffs_ex[1]*b2_ex}")
# Answer: (3, 2) → 3*(1,1) + 2*(1,-1) = (5, 1)